# Resumo Detalhado dos Dados PAM (Produção Agrícola Municipal)

Este notebook consolida o perfil dos dados da PAM, analisando variáveis críticas como área plantada, colhida e valor da produção por município.

In [2]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np

# Configuração de caminhos
# O notebook está em notebooks/etl-dados-necessarios/pam/data/resume/
# O dado está em notebooks/etl-dados-necessarios/pam/data/bronze/pam/
BASE_DIR = Path.cwd().parent.parent # notebooks/etl-dados-necessarios/pam/
PAM_BRONZE_DIR = BASE_DIR / "data/bronze/pam"
OUTPUT_DIR = BASE_DIR / "data/resume"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def analyze_pam_chunks(directory):
    files = list(directory.rglob("*.parquet"))
    if not files:
        print(f"❌ Nenhum arquivo .parquet encontrado em {directory}.")
        return None
    
    print(f"🔍 Analisando {len(files)} chunks da PAM...")
    
    # Lendo o primeiro arquivo para pegar a estrutura
    sample_df = pd.read_parquet(files[0])
    cols = sample_df.columns
    num_rows_total = 0
    
    # Dicionário para acumular métricas
    col_summary = {col: {"null_count": 0, "unique_values": set(), "min": None, "max": None, "dtype": str(sample_df[col].dtype)} for col in cols}
    
    for file in tqdm(files, desc="Processando Chunks"):
        df = pd.read_parquet(file)
        num_rows_total += len(df)
        
        for col in cols:
            series = df[col]
            col_summary[col]["null_count"] += int(series.isna().sum())
            
            # Para PAM, os valores únicos de categorias (Cultura, Município, Ano) são cruciais
            if len(col_summary[col]["unique_values"]) < 1000: # Limite para não estourar memória
                # Garante que adicionamos como string para evitar problemas de tipos mistos no set
                col_summary[col]["unique_values"].update(series.dropna().unique().astype(str).tolist())
            
            # Min/Max para numéricos e datas
            # Usando pd.api.types para evitar erro com StringDtype no numpy
            if pd.api.types.is_numeric_dtype(series.dtype):
                c_min = series.min()
                c_max = series.max()
                if col_summary[col]["min"] is None or (c_min is not None and c_min < col_summary[col]["min"]):
                    col_summary[col]["min"] = c_min
                if col_summary[col]["max"] is None or (c_max is not None and c_max > col_summary[col]["max"]):
                    col_summary[col]["max"] = c_max

    # Consolidando resultados
    final_summary = []
    for col, s in col_summary.items():
        final_summary.append({
            "column": col,
            "dtype": s["dtype"],
            "total_rows": num_rows_total,
            "null_count": s["null_count"],
            "null_percent": round((s["null_count"] / num_rows_total) * 100, 2) if num_rows_total > 0 else 0,
            "unique_values_count": len(s["unique_values"]),
            "min": s["min"],
            "max": s["max"]
        })
    
    return pd.DataFrame(final_summary)

summary_df = analyze_pam_chunks(PAM_BRONZE_DIR)

if summary_df is not None:
    output_file = OUTPUT_DIR / "resumo_detalhado_pam.parquet"
    summary_df.to_parquet(output_file, index=False)
    print(f"\n✅ Resumo consolidado exportado para: {output_file}")
    display(summary_df)


🔍 Analisando 10 chunks da PAM...


Processando Chunks: 100%|██████████| 10/10 [00:00<00:00, 50.11it/s]


✅ Resumo consolidado exportado para: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/pam/data/resume/resumo_detalhado_pam.parquet


,column,dtype,total_rows,null_count,null_percent,unique_values_count,min,max
0,NC,str,888340,0,0.0,2,None,None
1,NN,str,888340,0,0.0,2,None,None
2,MC,str,888340,0,0.0,6,None,None
3,MN,str,888340,0,0.0,6,None,None
4,V,str,888340,0,0.0,11092,None,None
5,D1N,str,888340,0,0.0,5564,None,None
6,D2C,str,888340,0,0.0,11,None,None
7,D2N,str,888340,0,0.0,11,None,None
8,D3C,str,888340,0,0.0,6,None,None
9,D3N,str,888340,0,0.0,6,None,None


### Análise de Consistência (Série Temporal)
Diferente de dados de fiscalização, a PAM é uma série histórica. Vamos verificar se temos dados para todos os anos esperados e quais culturas são predominantes.

In [8]:
# Carregando uma amostra maior para análise de negócio
files = list(PAM_BRONZE_DIR.rglob("*.parquet"))
if files:
    # Lendo todos os chunks para validação completa da qualidade
    df_sample = pd.concat([pd.read_parquet(f) for f in files])
    df_clean = df_sample.copy()
    
    # 1. Limpeza de Metadados: Remover linhas onde a coluna 'V' contém 'Valor'
    df_clean = df_clean[df_clean['V'] != 'Valor']
    
    # 2. Conversão de Tipos
    # V (Valor) para numérico
    df_clean['V'] = pd.to_numeric(df_clean['V'], errors='coerce')
    # D3N (Ano) para int
    if 'D3N' in df_clean.columns:
        df_clean['D3N'] = pd.to_numeric(df_clean['D3N'], errors='coerce').astype('Int64')

    print("--- 🔍 ANÁLISE QUALITATIVA DA AMOSTRA (PAM BRONZE - FULL CHUNKS) ---")
    
    # 1. Top Localidades por Registro
    print("\n📍 Top 5 Localidades (D1N):")
    display(df_clean['D1N'].value_counts().head(5))

    # 2. Variáveis Presentes (D2N)
    print("\n📊 Variáveis de Estudo (D2N):")
    display(df_clean['D2N'].unique().tolist())

    # 3. Análise de Produção por Variável
    print("\n📈 Resumo Estatístico por Variável (D2N):")
    stats_by_var = df_clean.groupby('D2N')['V'].describe()
    display(stats_by_var)
    
    output_var_stats = OUTPUT_DIR / "resumo_stats_por_variavel_pam.parquet"
    stats_by_var.reset_index().to_parquet(output_var_stats, index=False)

    # 4. Verificação de Produtos (D4N)
    print("\n🌾 Verificação de Produtos (D4N):")
    unique_products = df_clean['D4N'].unique().tolist()
    print(f"Total de valores únicos em D4N: {len(unique_products)}")
    print(f"Exemplos: {unique_products[:10]}")
    
    # Busca por produtos reais (que não sejam totais ou categorias)
    df_reais = df_clean[~df_clean['D4N'].str.contains('Total|produto|lavoura', case=False, na=False)]
    if not df_reais.empty:
        top_culturas = df_reais['D4N'].value_counts().head(10).reset_index()
        top_culturas.columns = ['Produto', 'Contagem']
        print("\n🏆 Top 10 Produtos Individuais:")
        display(top_culturas)
        output_top = OUTPUT_DIR / "resumo_top_culturas_pam.parquet"
        top_culturas.to_parquet(output_top, index=False)
    else:
        print("⚠️ AVISO: Nenhum produto individual (ex: Soja, Milho) encontrado nos chunks atuais.")
        print("Os dados atuais parecem conter apenas agregados (Totais).")

    # 5. Validação de Anos
    if 'D3N' in df_clean.columns:
        print(f"\n📅 Período total coberto: {df_clean['D3N'].min()} até {df_clean['D3N'].max()}")

    print(f"\n✅ Resumos consolidados exportados para: {OUTPUT_DIR}")
else:
    print("⚠️ Arquivos não encontrados para análise.")

--- 🔍 ANÁLISE QUALITATIVA DA AMOSTRA (PAM BRONZE - FULL CHUNKS) ---

📍 Top 5 Localidades (D1N):


D1N
Alta Floresta D'Oeste - RO    160
Ariquemes - RO                160
Cabixi - RO                   160
Cacoal - RO                   160
Cerejeiras - RO               160
Name: count, dtype: int64


📊 Variáveis de Estudo (D2N):


['Área plantada',
 'Área plantada - percentual do total geral',
 'Área colhida',
 'Área colhida - percentual do total geral',
 'Quantidade produzida',
 'Rendimento médio da produção',
 'Valor da produção',
 'Valor da produção - percentual do total geral',
 'Área destinada à colheita',
 'Área destinada à colheita - percentual do total geral']


📈 Resumo Estatístico por Variável (D2N):


,count,mean,std,min,25%,50%,75%,max
D2N,,,,,,,,
Quantidade produzida,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rendimento médio da produção,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Valor da produção,104984.0,69381.227901,293489.852472,1.0,833.0,4941.0,38500.25,11478917.0
Valor da produção - percentual do total geral,104984.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0
Área colhida,104994.0,8604.226737,37899.610365,1.0,83.0,622.0,4070.00,1218591.0
Área colhida - percentual do total geral,104994.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0
Área destinada à colheita,50162.0,1098.495475,3238.193856,1.0,26.0,109.0,627.00,61830.0
Área destinada à colheita - percentual do total geral,50162.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0
Área plantada,54882.0,15570.155169,51469.682802,1.0,587.0,2445.0,10847.00,1225091.0



🌾 Verificação de Produtos (D4N):
Total de valores únicos em D4N: 1
Exemplos: ['Total']
⚠️ AVISO: Nenhum produto individual (ex: Soja, Milho) encontrado nos chunks atuais.
Os dados atuais parecem conter apenas agregados (Totais).

📅 Período total coberto: 2020 até 2024

✅ Resumos consolidados exportados para: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/pam/data/resume


In [4]:
if 'df_sample' in locals():
    print("Colunas disponíveis no sample:")
    print(df_sample.columns.tolist())
    print("\nTipos de dados:")
    print(df_sample.dtypes)
    print("\nPrimeiras 3 linhas:")
    display(df_sample.head(3))
else:
    print("df_sample não encontrado.")

Colunas disponíveis no sample:
['NC', 'NN', 'MC', 'MN', 'V', 'D1N', 'D2C', 'D2N', 'D3C', 'D3N', 'D4C', 'D4N', 'tipo_lavoura']

Tipos de dados:
NC              str
NN              str
MC              str
MN              str
V               str
D1N             str
D2C             str
D2N             str
D3C             str
D3N             str
D4C             str
D4N             str
tipo_lavoura    str
dtype: object

Primeiras 3 linhas:


,NC,NN,MC,MN,V,D1N,D2C,D2N,D3C,D3N,D4C,D4N,tipo_lavoura
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município,Variável (Código),Variável,Ano (Código),Ano,Produto das lavouras temporárias (Código),Produto das lavouras temporárias,temporaria
1,6,Município,1006,Hectares,4563,Alta Floresta D'Oeste - RO,109,Área plantada,2023,2023,0,Total,temporaria
2,6,Município,2,Percentual,100.00,Alta Floresta D'Oeste - RO,1000109,Área plantada - percentual do total geral,2023,2023,0,Total,temporaria
